In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
from newutils.data.datasets import load_train_subset, add_lags
from newutils.other.helpers import init_context
from oldutils.types import TimeRange

init_context()

In [5]:
HORIZON_HISTORY = TimeRange.YEAR
HORIZON_FORECAST = TimeRange.WEEK

COL_GAUGE_ID = "gauge_id"
TARGETS = ["q_mm_day"]

In [6]:
df = load_train_subset()

In [7]:
lags = range(1, HORIZON_HISTORY + 1)
lags_columns = ["prcp", "t_max", "t_min", "t_mean", "q_mm_day", "lvl_sm"]
df = add_lags(df, lags_columns, lags)

In [8]:
targets = []
df = add_lags(df, targets, range(1, HORIZON_FORECAST + 1), targets)
targets.extend(TARGETS)

In [9]:
df = df.dropna()

In [10]:
X = df.drop(columns=set(lags_columns + targets))
y = df[targets]
X_train, y_train = X, y

# Model

In [11]:
from dask.distributed import Client, LocalCluster

# from dask_cuda import LocalCUDACluster
import gc

try:
    client.close()
    cluster.close()
    print("Delete")
except NameError:
    print("Nothing")
    pass
gc.collect()

cluster = LocalCluster(dashboard_address=":8789", asynchronous=False)
# cluster = LocalCUDACluster(CUDA_VISIBLE_DEVICES="0", n_workers=1, dashboard_address=":8789")
client = Client(cluster, asynchronous=False)
# client = Client()

# print(cluster)
print(client)

Delete
<Client: 'tcp://127.0.0.1:38995' processes=4 threads=4, memory=7.66 GiB>


In [12]:
import dask.array as da
import pandas as pd
import numpy as np
import dask.dataframe as dd
import xgboost.dask as dxgb

# n_rows = 10000
# pdf = pd.DataFrame({"value": np.random.randn(n_rows)})

# df = dd.from_pandas(pdf, npartitions=10)
# print(f"Number of partitions: {df.npartitions}")


# def add_shifts(pdf):
#     pdf["shift1"] = pdf["value"].shift(1)
#     pdf["shift2"] = pdf["value"].shift(2)
#     return pdf


# df = df.map_partitions(add_shifts)
# df = df.dropna()

# X = df[["shift1", "shift2"]]
# y = df["value"]

dtrain = dxgb.DaskDMatrix(client, X_train, y_train)

params = {
    "objective": "reg:squarederror",
    "max_depth": 3,
    "eta": 0.1,
    "tree_method": "hist",
}

# params["device"] = "cuda"

output = dxgb.train(client, params=params, dtrain=dtrain, num_boost_round=10)

print("Training metrics and booster information:")
# print(output)

TypeError: 'coroutine' object is not iterable

In [ ]:
import gc

client.close()
cluster.close()
gc.collect()

2353

In [ ]:
import xgboost, dask, distributed, sys

print("xgboost:", xgboost.__version__)
print("dask:", dask.__version__)
print("distributed:", distributed.__version__)
print("python:", sys.version)

xgboost: 3.0.0
dask: 2024.12.1
distributed: 2024.12.1
python: 3.12.3 (main, Feb  4 2025, 14:48:35) [GCC 13.3.0]
